# Chat with kaydee-qwen2.5-7b-lora

Loads the private LoRA adapter from Hugging Face on top of 4-bit Qwen2.5-7B-Instruct and chats as Kaushik.

**Kaggle settings:** Accelerator = **GPU T4** (one is enough), Internet = **on**, secret `hf_token` (read access to the private repo).

Optional: attach the `kaydee647-chat-dataset` dataset to compare against real held-out replies.

In [ ]:
!pip install -q unsloth

In [ ]:
import json
import os
import random

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("hf_token")

from unsloth import FastLanguageModel  # after HF_TOKEN so the private repo download is authenticated

ADAPTER = "kaydee647/kaydee-qwen2.5-7b-lora"
DATA_DIR = "/kaggle/input/datasets/kaushikdey2026/kaydee647-chat-dataset"
SYSTEM = "You are Kaushik Dey, chatting with {friend} on Instagram. Reply the way Kaushik would."
MAX_TURNS = 10  # context window used in training

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(ADAPTER, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)


def reply(messages, max_new_tokens=128, temperature=0.8, top_p=0.9):
    """messages: [system, ...turns ending on user] -> Kaushik's reply (may be several lines)."""
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, max_length=None,
        do_sample=True, temperature=temperature, top_p=top_p,
    )
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


class Chat:
    """Multi-turn chat with Kaushik; keeps the last MAX_TURNS turns like training did."""

    def __init__(self, friend):
        self.friend, self.history = friend, []

    def send(self, text, **kw):
        self.history.append({"role": "user", "content": text})
        msgs = [{"role": "system", "content": SYSTEM.format(friend=self.friend)}] + self.history[-MAX_TURNS:]
        r = reply(msgs, **kw)
        self.history.append({"role": "assistant", "content": r})
        print(f"{self.friend}: {text}")
        for line in r.splitlines():
            print(f"  Kaushik: {line}")
        return r

## Quick probes

Same questions as the 3B review, 3 samples each, to compare.

In [ ]:
probes = [
    ("Swapnil Agarwal", "Aur btao kya chal rha h"),
    ("Avantika", "Hiii, kaise ho?"),
    ("Avantika", "Weekend pe kya plan hai?"),
    ("Swapnil Agarwal", "Bhai kal office aa rha?"),
    ("Aditya Balakrishnan", "Did you watch the new episode?"),
    ("Avantika", "I'm feeling really low today"),
    ("Swapnil Agarwal", "What do you do for work?"),
    ("Avantika", "Are you an AI?"),
    ("Swapnil Agarwal", "Explain what a transformer model is"),
]
for friend, text in probes:
    msgs = [{"role": "system", "content": SYSTEM.format(friend=friend)}, {"role": "user", "content": text}]
    print(f"\n[{friend}] {text}")
    for _ in range(3):
        print("   ->", reply(msgs).replace("\n", " / "))

## Multi-turn conversation

Edit the friend and messages, or call `chat.send(...)` in new cells to keep the conversation going.

In [ ]:
chat = Chat("Swapnil Agarwal")
for text in ["Oye", "Kya kar rha", "Chal movie chalte aaj", "Kaunsi dekhein?"]:
    chat.send(text)

## Interactive chat

Type messages in the input box. `/friend <name>` switches person (clears history), `/reset` clears history, `/quit` stops. Interactive sessions only; skip this cell in a committed run.

In [ ]:
chat = Chat("Swapnil Agarwal")
print(f"Chatting as Kaushik with {chat.friend}. /friend <name>, /reset, /quit")
while True:
    text = input(f"{chat.friend}> ").strip()
    if not text:
        continue
    if text == "/quit":
        break
    if text == "/reset":
        chat = Chat(chat.friend)
        print("(history cleared)")
    elif text.startswith("/friend "):
        chat = Chat(text[len("/friend "):].strip() or chat.friend)
        print(f"(now chatting with {chat.friend})")
    else:
        chat.send(text)

## Held-out comparison (optional)

Model reply vs. what Kaushik actually said, on threads the model never saw. Needs the dataset attached.

In [ ]:
val_path = f"{DATA_DIR}/val.jsonl"
if not os.path.exists(val_path):
    print("Attach the kaydee647-chat-dataset dataset to run this cell.")
else:
    rows = [json.loads(l) for l in open(val_path)]
    random.Random(7).shuffle(rows)
    for row in rows[:15]:
        print("-" * 60)
        for t in row["prompt"][1:][-4:]:
            who = "Kaushik" if t["role"] == "assistant" else "friend"
            print(f"  {who:>7}: {t['content']!r}")
        print(f"    MODEL: {reply(row['prompt'])!r}")
        print(f"   ACTUAL: {row['completion'][0]['content']!r}")